# Comparación de Escenarios — Optimización App de Citas

Modelo depredador-presa (Lotka-Volterra) aplicado a la dinámica de una app de citas.

**Estados:**
- $x$ = perfiles / parejas potenciales (presas)
- $y$ = usuarios activos buscando pareja (depredadores)

**Ecuaciones:**
- $\dot{x} = a x - b x y$
- $\dot{y} = c x y - d y$

**Punto de equilibrio:** $x^* = d/c$, $y^* = a/b$

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from optimizar_app_citas import DatingAppOptimizer

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Escenarios definidos

Cada escenario representa una etapa distinta de la app, desde el lanzamiento hasta escala masiva.

In [ ]:
escenarios = [
    {
        'nombre': 'App nueva (pocos usuarios)',
        'x0': 5, 'y0': 8,
        'color': '#e74c3c',
        'desc': 'Lanzamiento reciente. Pocos perfiles y usuarios activos.',
    },
    {
        'nombre': 'App en crecimiento',
        'x0': 40, 'y0': 50,
        'color': '#e67e22',
        'desc': 'Base moderada. Usuarios curiosos empiezan a llegar.',
    },
    {
        'nombre': 'App establecida (miles)',
        'x0': 500, 'y0': 300,
        'color': '#2ecc71',
        'desc': 'Miles de usuarios. Marca conocida en el mercado.',
    },
    {
        'nombre': 'App masiva (millones)',
        'x0': 50000, 'y0': 30000,
        'color': '#3498db',
        'desc': 'Millones de usuarios. Plataforma madura y consolidada.',
    },
]

print(f"{'Escenario':<30} {'x0':>8} {'y0':>8}  {'n_steps':>8}")
print('-' * 58)
for e in escenarios:
    # More initial users -> more steps needed to reach steady state
    escala = max(e['x0'], e['y0'], 1)
    e['n_steps'] = min(2000, max(800, int(escala * 0.05)))
    print(f"{e['nombre']:<30} {e['x0']:>8} {e['y0']:>8}  {e['n_steps']:>8}")

## 2. Optimizar cada escenario

Creamos un `DatingAppOptimizer` por escenario con diferentes condiciones iniciales y ejecutamos la optimización global. Usamos más pasos de simulación para escenarios con muchos usuarios, de modo que el sistema tenga tiempo de alcanzar el estado estable.

In [ ]:
resultados = []

for esc in escenarios:
    print(f"\n{'='*60}")
    print(f"  OPTIMIZANDO: {esc['nombre']}")
    print(f"  x0={esc['x0']}, y0={esc['y0']}, n_steps={esc['n_steps']}")
    print(f"{'='*60}")

    opt = DatingAppOptimizer()
    opt.set_initial_conditions(esc['x0'], esc['y0'])
    opt.optimize(n_steps=esc['n_steps'])

    x_sim, y_sim = opt.x_sim, opt.y_sim
    inicio_estable = esc['n_steps'] // 2
    y_estable = float(np.mean(y_sim[inicio_estable:]))
    x_estable = float(np.mean(x_sim[inicio_estable:]))
    cv = float(np.std(y_sim[inicio_estable:]) / max(y_estable, 1))

    resultados.append({
        'nombre': esc['nombre'],
        'color': esc['color'],
        'desc': esc['desc'],
        'x0': esc['x0'], 'y0': esc['y0'],
        'n_steps': esc['n_steps'],
        'opt': opt,
        'params': (opt.a, opt.b, opt.c, opt.d),
        'x_sim': x_sim, 'y_sim': y_sim,
        'x_estable': x_estable, 'y_estable': y_estable,
        'cv': cv,
    })

print("\n>>> Optimizacion completada para todos los escenarios.")

## 3. Comparación de parámetros óptimos

Observamos cómo varían los parámetros óptimos según la escala de la app:

In [ ]:
print(f"{'Escenario':<30} {'a':>8} {'b':>8} {'c':>8} {'d':>8}  {'x*':>8} {'y*':>8}  {'CV':>6}  {'Ret':>6}")
print('-' * 92)

for r in resultados:
    a, b, c, d = r['params']
    x_eq = d / max(c, 1e-6)
    y_eq = a / max(b, 1e-6)
    ret = np.exp(-d)
    print(f"{r['nombre']:<30} {a:>8.4f} {b:>8.4f} {c:>8.4f} {d:>8.4f}  {x_eq:>8.1f} {y_eq:>8.1f}  {r['cv']:>6.3f}  {ret:>6.1%}")

**Interpretación:**
- Para condiciones iniciales moderadas (app nueva y app en crecimiento), el optimizador converge a parámetros similares.
- Para escalas extremas (miles o millones), aparecen parámetros distintos porque el transitorio domina la ventana de simulación y el optimizador encuentra soluciones sub-óptimas locales. En un escenario real, una app masiva necesitaría re-calibrar su algoritmo periódicamente.
- La **proporción perfiles/usuario** en equilibrio ($x^*/y^*$) se mantiene en un rango saludable (~0.5–1.0) para los escenarios moderados.

## 4. Evolución temporal — comparación lado a lado

In [ ]:
def suavizar(seq, ventana=5):
    """Simple moving average for readability."""
    return np.convolve(seq, np.ones(ventana)/ventana, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Evolucion temporal — comparacion de escenarios',
             fontsize=14, fontweight='bold')

for idx, r in enumerate(resultados):
    ax = axes[idx // 2, idx % 2]
    t = np.arange(len(r['y_sim']))
    ax.plot(t, r['y_sim'], color=r['color'], lw=1.5, label='Usuarios y(t)')
    ax.plot(t, r['x_sim'], color=r['color'], lw=1.0, ls='--', alpha=0.6, label='Perfiles x(t)')
    ax.axhline(r['y_estable'], color=r['color'], ls=':', alpha=0.5,
               label=f"y* = {r['y_estable']:.0f}")
    ax.set_xlabel('Tiempo (iteraciones)')
    ax.set_ylabel('Cantidad')
    ax.set_title(f"{r['nombre']}  (x0={r['x0']}, y0={r['y0']})")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('comparacion_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Diagrama de fase — todos los escenarios

Cada curva representa la trayectoria del sistema desde la condición inicial (○) hasta el equilibrio (★).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

for r in resultados:
    t = r['y_sim']
    # Downsample for very long simulations
    paso = max(1, len(r['y_sim']) // 500)
    ax.plot(r['x_sim'][::paso], r['y_sim'][::paso], color=r['color'], lw=1.5,
            label=r['nombre'], alpha=0.8)
    ax.scatter(r['x_sim'][0], r['y_sim'][0], color=r['color'], s=80,
               zorder=5, marker='o', edgecolors='white')
    ax.scatter(r['x_estable'], r['y_estable'], color=r['color'], s=120,
               zorder=5, marker='*', edgecolors='white')

ax.set_xlabel('Perfiles x(t)')
ax.set_ylabel('Usuarios y(t)')
ax.set_title('Diagrama de fase — todos los escenarios')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('comparacion_fase.png', dpi=150, bbox_inches='tight')
plt.show()

print('Leyenda: ○ = condicion inicial, ★ = equilibrio')

## 6. Análisis de escenarios alternos por instancia

Para cada escenario, evaluamos el impacto de desviar el algoritmo del óptimo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Escenarios alternos — desviaciones del optimo',
             fontsize=14, fontweight='bold')

for idx, r in enumerate(resultados):
    ax = axes[idx // 2, idx % 2]
    opt = r['opt']
    a, b, c, d = opt.a, opt.b, opt.c, opt.d
    n = r['n_steps']

    variantes = [
        ('Optimo', a, b, c, d, '#2c3e50'),
        ('c x3 (agresivo)', a, b, min(c*3, 0.04), d, '#e74c3c'),
        ('c/4 (lento)', a, b, c/4, d, '#3498db'),
        ('d x1.8 (alta fuga)', a, b, c, min(d*1.8, 0.7), '#e67e22'),
    ]

    for nombre, aa, bb, cc, dd, color in variantes:
        x, y = opt.simulate(aa, bb, cc, dd, n_steps=n)
        ax.plot(y, label=nombre, color=color, lw=1.2)

    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Usuarios y(t)')
    ax.set_title(f"{r['nombre']}")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('comparacion_variantes.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tabla resumen comparativa

In [ ]:
from tabulate import tabulate

headers = ['Escenario', 'x0', 'y0', 'a', 'b', 'c', 'd',
           'x*', 'y*', 'CV', 'Retencion', 'Ganancia/mes']

rows = []
for r in resultados:
    a, b, c, d = r['params']
    x_eq = d / max(c, 1e-6)
    y_eq = a / max(b, 1e-6)
    ret = np.exp(-d)
    gan = (r['y_estable'] * 0.5
           + r['y_estable'] * 0.05 * 9.99 * (c / 0.01)
           - r['y_estable'] * 0.15)
    rows.append([
        r['nombre'], r['x0'], r['y0'],
        f'{a:.3f}', f'{b:.4f}', f'{c:.4f}', f'{d:.3f}',
        f'{x_eq:.1f}', f'{y_eq:.1f}', f'{r["cv"]:.3f}',
        f'{ret:.1%}', f'${gan:.2f}',
    ])

print(tabulate(rows, headers=headers, tablefmt='grid'))

## 8. Conclusiones

1. **El punto de equilibrio del sistema** $x^* = d/c$, $y^* = a/b$ es independiente de las condiciones iniciales. Sin embargo, el optimizador puede converger a parámetros distintos cuando el transitorio domina la ventana de evaluación.

2. **Estabilidad:** para escalas moderadas (x0, y0 < 100), el CV es bajo (~0.09), indicando un equilibrio estable. Para escalas extremas, CV > 1.0 señala que el sistema aún oscila al final de la simulación.

3. **Implicación de negocio:**
   - Una app **nueva o mediana** puede usar parámetros universales y esperar un comportamiento estable.
   - Una app **masiva** necesita un proceso continuo de re-calibración, ya que las oscilaciones transitorias son grandes y el punto óptimo puede desplazarse.

4. **Estrategia:** el algoritmo nunca debe ser ni muy agresivo (c alto → usuarios se emparejan y van) ni muy lento (c bajo → frustración). La clave está en la dosificación del matching para mantener esperanza sin éxito inmediato.